In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import( root_mean_squared_error,mean_absolute_error,r2_score)
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_validate
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

In [ ]:
df=pd.read_csv("housing.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.columns:
    print(df[col].value_counts().head(20))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.countplot(x='ocean_proximity',data=df)
#plt.xticks(rotation=90)

plt.show()
print(df['ocean_proximity'].value_counts())

In [ ]:
sns.histplot(df["median_house_value"],bins=30,kde=True)
plt.show()

In [ ]:
df['median_house_value'].value_counts()


In [ ]:
num_cols=df.select_dtypes(include='number').columns

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(15,10))
axes=axes.flatten()
for i,col in enumerate(num_cols):
   sns.histplot(df[col],kde=True,ax=axes[i])
   axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig,axes=plt.subplots(3,3,figsize=(15,10))
axes=axes.flatten()
for i,col in enumerate(num_cols):
   sns.boxplot(df[col],ax=axes[i])
   axes[i].set_title(col,fontsize=8)
   axes[i].set_xlabel(" ")
    
plt.tight_layout()
plt.show()

In [ ]:
#heatmap-used for showing correlation b/w data
plt.figure(figsize=(10,5))
sns.heatmap(
    df[num_cols].corr(),
    annot=True,
    cmap="coolwarm",center=0
)
plt.title("correlated heatmap")
plt.show()


In [ ]:
corr_with_target=df[num_cols].corr()['median_house_value'].sort_values(ascending=False)
print(corr_with_target)

Key Insights from EDA
>One categorical and rest numeric values
>total_bedrooms has missing values
>target is right skewed
>income is strongest predictor
>multicollinearity among rooms and population

Preprocessing & Evaluation Plan

• Median imputation for missing values
• One-hot encoding for categorical feature
• Feature scaling for linear models
• Use pipelines to avoid data leakage
• Baseline model → model selection → hyperparameter tuning
• Primary metric: RMSE, secondary: MAE and R²
• Final evaluation only on test set

In [ ]:
X=df.drop(columns="median_house_value")
y=df["median_house_value"]

In [ ]:
X_train,X_test,y_train,y_test= train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [ ]:
numerical_feature=X_train.select_dtypes(include=[np.number]).columns.tolist()
numerical_feature

In [ ]:
categorical_feature=X_train.select_dtypes(exclude=[np.number]).columns.tolist()
categorical_feature

In [ ]:
#numeric and categorical features preprocessing
numeric_transformer=Pipeline(
    steps=[("imputer",SimpleImputer(strategy="median")),
             ("scaler",StandardScaler())]
)
categoric_transformer=Pipeline(
    steps=[("imputer",SimpleImputer(strategy="most_frequent")),
           ("onehot",OneHotEncoder(handle_unknown="ignore"))]
)
#preprocessing pipeline
preprocess=ColumnTransformer([
    ("nums",numeric_transformer,numerical_feature),
    ("cats",categoric_transformer,categorical_feature)
])

In [ ]:
#BASE LINE MODEL(no CV,no tuning)
baseline_pipe=Pipeline(
    steps=[
        ("preprocess",preprocess),
        ("model",LinearRegression())
    ]
)

In [ ]:
#preprocess data and train baseline pipeline
baseline_pipe.fit(X_train,y_train)

In [ ]:
print(type(numeric_transformer))
print(type(categoric_transformer))
print(type(numerical_feature))
print(type(categorical_feature))

In [ ]:
train_baseline_pred=baseline_pipe.predict(X_train)
test_baseline_pred=baseline_pipe.predict(X_test)

In [ ]:
train_baseline_pred[:5]

In [ ]:
train_baseline_rmse=root_mean_squared_error(y_train,train_baseline_pred)
train_baseline_mae=mean_absolute_error(y_train,train_baseline_pred)
train_baseline_r2=r2_score(y_train,train_baseline_pred)

In [ ]:
y_train[:5]

In [ ]:
train_baseline_rmse

In [ ]:
#these are the models to try
models={
    "LinearRegression":LinearRegression(),
    "Ridge":Ridge(random_state=42),
    "LassoRegression":Lasso(random_state=42,max_iter=10000),
    "RandomForest":RandomForestRegressor(),
    "HistGB":HistGradientBoostingRegressor(random_state=42)
    }

In [ ]:
k=5
cv=KFold(n_splits=k,shuffle=True,random_state=42)

In [ ]:
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2"
}

In [ ]:
rows=[]
for name,model in models.items():
    pipe=Pipeline(
        steps=[
            ("preprocess",preprocess),
            ("model",model)
        ]
    )
    scores=cross_validate(pipe,X_train,y_train,cv=cv,scoring=scoring,n_jobs=1)
    rows.append({
        "model":name,
        "cv_rmse":-scores["test_rmse"].mean(),
        "cv_mae":-scores["test_mae"].mean(),
        "cv_r2":-scores["test_r2"].mean()
        
        
    })
cv_results=pd.DataFrame(rows).sort_values("cv_rmse")


In [ ]:
scores

In [ ]:
cv_results

In [ ]:
#hyperparameter tuning
histgb_pipe=Pipeline(
    steps=[("preprocess",preprocess),
            ("model",HistGradientBoostingRegressor(random_state=42))]
)

In [ ]:
# hyperparameters combination
param_grid = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_depth": [None, 3, 6],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [20, 50, 100],
    "model__l2_regularization": [0.0, 0.1, 1.0]
}

In [ ]:
grid = GridSearchCV(
    estimator=histgb_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid.fit(X_train,y_train)

In [ ]:
-grid.best_score_

In [ ]:
grid.best_params_

In [ ]:
#retrain with best parameter
histgb_best=Pipeline(
    steps=[("preprocess",preprocess),
           ("model",HistGradientBoostingRegressor(l2_regularization=0.1,
                                                  learning_rate=0.1,
                                                  max_depth=None,
                                                  max_leaf_nodes=63,
                                                  min_samples_leaf=20))
])

In [ ]:
histgb_best.fit(X_train,y_train)

In [ ]:
#final evaluation 
train_final_predict=histgb_best.predict(X_train)
train_final_rmse=root_mean_squared_error(y_train,train_final_predict)
train_final_mae=mean_absolute_error(y_train,train_final_predict)
train_final_r2=r2_score(y_train,train_final_predict)

In [ ]:
print(train_final_rmse)
print(train_final_mae)
print(train_final_r2)

In [ ]:
def predict_house_price(
    model,
    longitude: float,
    latitude: float,
    housing_median_age: float,
    total_rooms: float,
    total_bedrooms: float,
    population: float,
    households: float,
    median_income: float,
    ocean_proximity: str
) -> float:

    #Predict median_house_value for one new house.
    #total_bedrooms can be np.nan (pipeline will impute).

    new_row = pd.DataFrame([{
        "longitude": longitude,
        "latitude": latitude,
        "housing_median_age": housing_median_age,
        "total_rooms": total_rooms,
        "total_bedrooms": total_bedrooms,
        "population": population,
        "households": households,
        "median_income": median_income,
        "ocean_proximity": ocean_proximity
    }])
    return float(model.predict(new_row)[0])

In [ ]:
sample_price = predict_house_price(
    model=histgb_best,
    longitude=-122.23,
    latitude=37.88,
    housing_median_age=41,
    total_rooms=880,
    total_bedrooms=129,
    population=322,
    households=126,
    median_income=8.3252,
    ocean_proximity="NEAR BAY"
)

print(sample_price)